# FAOSTAT Pesticide Use Preprocessing

Aggregates annual FAOSTAT pesticide indicators from 2015 onward.

In [ ]:
import re
import pandas as pd
from paths import PREPROCESSED_PATHS, RAW_PATHS, save_dataset


In [ ]:
def sanitize(*parts):
    text = '_'.join(str(p).strip() for p in parts if str(p).strip())
    text = re.sub(r'[^\w]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

raw_files = sorted(RAW_PATHS['faostat_dir'].glob('FAOSTAT_data_en_*.csv'))
frames = [pd.read_csv(path, encoding='latin-1') for path in raw_files]
raw = pd.concat(frames, ignore_index=True).drop_duplicates()
raw = raw[raw['Year'] >= 2015].copy()
raw['Value'] = pd.to_numeric(raw['Value'], errors='coerce')
raw['Feature'] = raw.apply(lambda r: sanitize('FAO', r['Item'], r['Element'], r.get('Unit', '')), axis=1)
annual = raw.pivot_table(index='Year', columns='Feature', values='Value', aggfunc='mean')
save_dataset(annual, PREPROCESSED_PATHS['faostat_annual'])
print('FAOSTAT annual shape:', annual.shape)
